In [27]:
# final_merged_model.py
from transformers import AutoModelForSeq2SeqLM, MBart50TokenizerFast
from peft import PeftModel
import torch

# 1. Merge and Save Function (Run Once)
def merge_and_save():
    # Load base model and original tokenizer
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        "facebook/mbart-large-50",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    # Load original tokenizer with language settings
    tokenizer = MBart50TokenizerFast.from_pretrained(
        "facebook/mbart-large-50",
        src_lang="en_XX",
        tgt_lang="hi_IN"
    )

    # Load trained LoRA adapter
    peft_model = PeftModel.from_pretrained(base_model, "model_with_lora4")
    
    # Merge and convert to float16 for stability
    merged_model = peft_model.merge_and_unload()
    merged_model = merged_model.to(torch.float16)

    # Save merged components
    merged_model.save_pretrained("merged_mbart")
    tokenizer.save_pretrained("merged_mbart")  # Critical for language codes

# 2. Inference Function
def load_and_translate(text):
    # Load merged model and tokenizer
    model = AutoModelForSeq2SeqLM.from_pretrained(
        "merged_mbart",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    tokenizer = MBart50TokenizerFast.from_pretrained(
        "merged_mbart",
        src_lang="en_XX",
        tgt_lang="hi_IN"
    )

    # Verify language settings
    tokenizer.src_lang = "en_XX"
    tokenizer.tgt_lang = "hi_IN"
    model.config.forced_bos_token_id = tokenizer.lang_code_to_id["hi_IN"]

    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)

    # Generate translation
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"],
        max_new_tokens=512,
        num_beams=5,
        temperature=0.7,
        early_stopping=True
    )

    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]




In [ ]:
# Execution Flow
if __name__ == "__main__":
    # Step 1: Merge once (comment after first run)
    # merge_and_save()
    
    # Step 2: Test translations
    test_phrases = [
        "Hello, how are you?"
    ]
    
    for phrase in test_phrases:
        translation = load_and_translate(phrase)
        print(f"Input: {phrase}")
        print(f"Translation: {translation}\n")

In [1]:
import random
import evaluate
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline, AutoModelForSequenceClassification
from tqdm import tqdm
import pandas as pd
from datasets import Dataset, load_dataset
from openai import OpenAI


/home/deepan/AnanditaUma/translation/translation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:

# Load Evaluation Metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
perplexity_metric = evaluate.load("perplexity")
bertscore = evaluate.load("bertscore")

# --- Generalized Sampling Function ---
def sample_data(dataset, n=50):
    dataset_list = list(dataset)
    return random.sample(dataset_list, min(n, len(dataset_list)))

# --- Task-Specific Inference Helpers ---
def slm_inference(model_pipeline, inputs, batch_size=8, task_type="classification"):
    results = []
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i+batch_size]
        if task_type == "classification":
            batch_results = model_pipeline(batch, truncation=True, padding=True)
            results.extend([int(res["label"].replace("LABEL_", "")) for res in batch_results])
        elif task_type == "generation":
            generated = model_pipeline(batch, max_length=100)
            results.extend([g[0]["generated_text"] for g in generated])
    return results

def llm_inference(api_call_func, inputs, batch_size=8):
    results = []
    for i in range(0, len(inputs), batch_size):
        results.extend(api_call_func(inputs[i:i+batch_size]))
    return results

# --- Generalized Metric Calculation ---
def calculate_metrics(predictions, references, task_type):
    results = {}
    if task_type == "generation":
        # Existing metrics
        results['BLEU'] = bleu.compute(predictions=predictions, references=[[ref] for ref in references])["bleu"]
        results['ROUGE-L'] = rouge.compute(predictions=predictions, references=references)["rougeL"]
        results['Perplexity'] = perplexity_metric.compute(predictions=predictions, model_id="gpt2")["perplexities"][0]

        # New: BERTScore
        bertscore_results = bertscore.compute(predictions=predictions, references=references, lang="en")
        results['BERTScore_P'] = sum(bertscore_results['precision']) / len(bertscore_results['precision'])
        results['BERTScore_R'] = sum(bertscore_results['recall']) / len(bertscore_results['recall'])
        results['BERTScore_F1'] = sum(bertscore_results['f1']) / len(bertscore_results['f1'])
    elif task_type == "classification":
        results['Accuracy'] = accuracy_score(references, predictions)
        results['F1-Score'] = f1_score(references, predictions, average='weighted')
    return results


# --- Model Loading Helpers ---
def load_slm(model_path, task_type):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if task_type == "classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        return pipeline("text-classification", model=model, tokenizer=tokenizer)
    elif task_type == "generation":
        return pipeline("text-generation", model=model_path, tokenizer=tokenizer)

# --- Generalized Evaluation Function ---
def evaluate_task(models, dataset, task_config):
    sampled_data = sample_data(dataset)
    inputs = [item[task_config["input_key"]] for item in sampled_data]
    references = [item[task_config["ref_key"]] for item in sampled_data]
    
    results = []
    for name, model in models.items():
        if task_config["model_type"] == "slm":
            preds = slm_inference(model, inputs, task_type=task_config["task_type"])
        else:
            preds = llm_inference(model, inputs)
        
        metrics = calculate_metrics(preds, references, task_config["task_type"])
        results.append({"Model": name, **metrics})
    
    return pd.DataFrame(results)

# === Task Configurations ===
TASK_CONFIGS = {
    "sentiment": {
        "input_key": "text",
        "ref_key": "label",
        "task_type": "classification",
        "slm_paths": {
            "pre": "roberta-base",
            "post": "/home/deepan/AnanditaUma/sentimentAnalysis/sentimentanalysis"
        }
    },
    "translation": {
        "input_key": "english_sentence",
        "ref_key": "hindi_sentence",
        "task_type": "generation",
        "slm_paths": {
            "pre": "mbart-large-50",
            "post": "/home/deepan/AnanditaUma/translation/model_with_lora4/"
        }
    },
    "content": {
        "input_key": "prompt",
        "ref_key": "story",
        "task_type": "generation",
        "slm_paths": {
            "pre": "gpt2",
            "post": "/home/deepan/AnanditaUma/contentcreation/gpt2-writingprompts2/"
        }
    }
}

# --- Task Evaluation Functions ---
def evaluate_slm(task_name, test_set):
    config = TASK_CONFIGS[task_name]
    models = {
        "SLM Pre": load_slm(config["slm_paths"]["pre"], config["task_type"]),
        "SLM Post": load_slm(config["slm_paths"]["post"], config["task_type"])
    }
    return evaluate_task(models, test_set, {**config, "model_type": "slm"})

def evaluate_llm(task_name, test_set, api_functions):
    config = TASK_CONFIGS[task_name]
    models = {name: fn for name, fn in api_functions.items()}
    return evaluate_task(models, test_set, {**config, "model_type": "llm"})


In [ ]:
full_test_set_sa = load_dataset("yelp_review_full", split="train")
test_set_sa = full_test_set_sa.shuffle(seed=42).select(range(50))
full_test_set_trans = pd.read_csv("testing_corpus.csv")
full_test_set_trans_hf = Dataset.from_pandas(full_test_set_trans)
test_set_trans = full_test_set_trans_hf.shuffle(seed=42).select(range(5000))
sentiment_results = evaluate_slm("sentiment", test_set_sa)
print(sentiment_results)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
Device set to use cuda:0


      Model  Accuracy  F1-Score
0   SLM Pre      0.24  0.092903
1  SLM Post      0.72  0.715714


In [3]:
full_test_set_sa = load_dataset("yelp_review_full", split="train")
test_set_sa = full_test_set_sa.shuffle(seed=42).select(range(100))
full_test_set_trans = pd.read_csv("testing_corpus.csv")
full_test_set_trans_hf = Dataset.from_pandas(full_test_set_trans)
test_set_trans = full_test_set_trans_hf.shuffle(seed=42).select(range(5000))

In [64]:
sentiment_results = evaluate_slm("sentiment", test_set_sa)
print(sentiment_results)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
Device set to use cuda:0


      Model  Accuracy  F1-Score
0   SLM Pre      0.24  0.092903
1  SLM Post      0.74  0.737333


In [47]:
sentiment_results = evaluate_slm("sentiment", test_set_sa)
print(sentiment_results)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
Device set to use cuda:0


      Model  Accuracy  F1-Score
0   SLM Pre      0.26  0.107302
1  SLM Post      0.80  0.803626


In [4]:
test_set_sa_llm = full_test_set_sa.shuffle(seed=42).select(range(50))
# Filter out non-dictionary items and ensure structure
cleaned_test_set = []
for item in test_set_sa_llm:
    if isinstance(item, dict) and "text" in item:
        cleaned_test_set.append(item)
    else:
        print(f"Removing invalid entry: {item}")

test_set_sa_llm = cleaned_test_set  # Use cleaned dataset


In [5]:
import openai


In [28]:

# Prompt for the model
system_prompt = """
You are an English sentiment classifier.
Return ONLY a single integer from 0 to 4 based on this scale:
0: Very negative
1: Negative
2: Neutral
3: Positive
4: Very positive
Do not include any explanation or additional text.
"""

# Init OpenAI client with OpenRouter
ds_client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")

# Define the robust sentiment function
def safe_deepseek_sentiment(batch):
    responses = []
    for text in batch:
        try:
            response = ds_client.chat.completions.create(
                model="mistralai/mistral-small-3.1-24b-instruct:free",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            # Log full raw response for debugging
            logger.info(f"Raw response: {content}")

            # Extract integer (0 to 4) from the model response
            match = re.search(r"\b[0-4]\b", content)
            if match:
                label = int(match.group(0))
                responses.append(label)
            else:
                raise ValueError(f"No valid sentiment label found in response: '{content}'")

        except Exception as e:
            logger.error(f"Error processing input {len(responses)}: {text}\nException: {e}")
            responses.append(2)  # Default to Neutral
            time.sleep(1)
    return responses

# Hook it up to your eval
sentiment_llms = {
    "DeepSeek": safe_deepseek_sentiment
}

# Assuming test_set_sa_llm and evaluate_llm are defined in your codebase
sentiment_results_llm = evaluate_llm("sentiment", test_set_sa_llm, sentiment_llms)
print(sentiment_results_llm)


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 1
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 2
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 1
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 3
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 4
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 1
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 

      Model  Accuracy  F1-Score
0  DeepSeek      0.52  0.532738


In [31]:

# Prompt for the model
system_prompt = (
    "You are a sentiment classifier. Read the review and respond with ONLY a single integer (0–4) based on the following:\n"
    "0 = Very Negative, 1 = Negative, 2 = Neutral, 3 = Positive, 4 = Very Positive.\n"
    "Reply with only the integer."
)



# Init OpenAI client with OpenRouter
ds_client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")

# Define the robust sentiment function
def safe_deepseek_sentiment(batch):
    responses = []
    for text in batch:
        try:
            response = ds_client.chat.completions.create(
                model="qwen/qwen3-235b-a22b:free",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            # Log full raw response for debugging
            logger.info(f"Raw response: {content}")

            # Extract integer (0 to 4) from the model response
            match = re.search(r"\b[0-4]\b", content)
            if match:
                label = int(match.group(0))
                responses.append(label)
            else:
                raise ValueError(f"No valid sentiment label found in response: '{content}'")

        except Exception as e:
            logger.error(f"Error processing input {len(responses)}: {text}\nException: {e}")
            responses.append(2)  # Default to Neutral
            time.sleep(1)
    return responses

# Hook it up to your eval
sentiment_llms = {
    "DeepSeek": safe_deepseek_sentiment
}

# Assuming test_set_sa_llm and evaluate_llm are defined in your codebase
sentiment_results_llm = evaluate_llm("sentiment", test_set_sa_llm, sentiment_llms)
print(sentiment_results_llm)


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 4
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 1
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 2
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: 4
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 

      Model  Accuracy  F1-Score
0  DeepSeek       0.6  0.575883


In [ ]:
# Prompt for the model
system_prompt = (
    "You are a sentiment classifier. Read the review and respond with ONLY a single integer (0–4) based on the following:\n"
    "0 = Very Negative, 1 = Negative, 2 = Neutral, 3 = Positive, 4 = Very Positive.\n"
    "Reply with only the integer."
)

# Init OpenAI client with OpenRouter
ds_client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="your-api-key"
)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")

# Define the robust sentiment function
def safe_deepseek_sentiment(batch):
    responses = []
    for text in batch:
        try:
            response = ds_client.chat.completions.create(
                model="microsoft/phi-4-reasoning-plus:free",
                messages=[ 
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            # Log the response for debugging (optional)
            logger.info(f"Raw response: {content}")

            # Extract integer (0 to 4) from the model response
            match = re.search(r"\b[0-4]\b", content)
            if match:
                label = int(match.group(0))
                responses.append(label)
            else:
                raise ValueError(f"No valid sentiment label found in response: '{content}'")

        except Exception as e:
            logger.error(f"Error processing input {len(responses)}: {text}\nException: {e}")
            responses.append(2)  # Default to Neutral
            time.sleep(1)
    return responses

# Assuming test_set_sa_llm is defined in your codebase
sentiment_llms = {
    "DeepSeek": safe_deepseek_sentiment
}

# Test the function and print the result
sentiment_results_llm = evaluate_llm("sentiment", test_set_sa_llm, sentiment_llms)
print(sentiment_results_llm)


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"
ERROR:LLM_Evaluation:Error processing input 0: Your review text here
Exception: Error code: 401 - {'error': {'message': 'No auth credentials found', 'code': 401}}


[2]


In [44]:
full_test_set_cc = load_dataset("euclaise/writingprompts", split="test")
test_set_cc = full_test_set_cc.shuffle(seed=42).select(range(10000))
test_set_cc_llm = full_test_set_cc.shuffle(seed=42).select(range(50))



In [45]:
content_results = evaluate_slm("content", test_set_cc)
print(content_results)

Device set to use cuda:0
Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
S

      Model      BLEU   ROUGE-L  Perplexity  BERTScore_P  BERTScore_R  \
0   SLM Pre  0.000075  0.076155   17.287394     0.805742     0.777866   
1  SLM Post  0.000159  0.082801   35.121326     0.821226     0.793773   

   BERTScore_F1  
0      0.791486  
1      0.807196  


In [46]:

# Prompt for the model
system_prompt = (
    "You are a short story genrator. read the prompt given and based on that generate a short story. It should not be very long."
)

# Init OpenAI client with OpenRouter
ds_client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LLM_Evaluation")

# Define the robust sentiment function
def safe_deepseek_content(batch):
    responses = []
    for text in batch:
        
            response = ds_client.chat.completions.create(
                model="mistralai/mistral-small-3.1-24b-instruct:free",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"{text}"}
                ]
            )
            content = response.choices[0].message.content.strip()
            # Log full raw response for debugging
            logger.info(f"Raw response: {content}")

    return responses

# Hook it up to your eval
content_llms = {
    "DeepSeek": safe_deepseek_content
}

# Assuming test_set_sa_llm and evaluate_llm are defined in your codebase
content_results_llm = evaluate_llm("content", test_set_cc_llm, content_llms)
print(content_results_llm)


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:LLM_Evaluation:Raw response: In the bustling city of Neo-Crest, the unemployment rate had soared to an unprecedented 30%. The government, desperate to revitalize the economy, began creating new jobs—jobs that some deemed useless, but others found strangely fulfilling.

One such job was the " Urban Greeter." Emily, a former marketing professional, found herself in this unusual role. Her job was to stand at various points in the city, greeting passersby with a cheerful "Good morning!" or "Have a lovely day!" She was provided with a bright yellow vest and a name tag that read "Emily—Your Urban Greeter."

At first, Emily felt silly. She'd stand at her designated spot, grinning like a maniac as people rushed past, dodging her attempts to make eye contact. But as days turned into weeks, something changed. Strangers started to wave back. Some even stopped to chat. A few regulars began to look fo

IndexError: list index out of range

In [7]:
import re
import time
from concurrent.futures import ThreadPoolExecutor
import logging